In [1]:
import pandas as pd 

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

In [4]:
fd = pd.read_csv('../Week-3/Data/creditcard.csv')

# Exploring data

In [5]:
print("shape: ", fd.shape)

shape:  (284807, 31)


In [6]:
print("Columns: ", fd.columns.to_list())

Columns:  ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [7]:
print("First 5 rows:")
fd.head()

First 5 rows:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


 # Looking for the imbalance 

In [8]:
print("Class distribution:")
print(fd['Class'].value_counts())
print('Fraud percentage:')
print(f"{fd['Class'].mean() * 100:.3f}%")

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64
Fraud percentage:
0.173%


# BAsics statistics

In [9]:
print("Dataset Info:")
print(f"Time range: {fd['Time'].min():.0f} to {fd['Time'].max():.0f} seconds")
print(f"Amount range: ${fd['Amount'].min():.2f} to ${fd['Amount'].max():.2f}")
print(f"Average transaction: ${fd['Amount'].mean():.2f}")
print(f"Average fraud amount: ${fd[fd['Class']==1]['Amount'].mean():.2f}")
print(f"Average normal amount: ${fd[fd['Class']==0]['Amount'].mean():.2f}")

Dataset Info:
Time range: 0 to 172792 seconds
Amount range: $0.00 to $25691.16
Average transaction: $88.35
Average fraud amount: $122.21
Average normal amount: $88.29


In [10]:
print("Shape:", fd.shape)
print("Fraud cases:", fd['Class'].sum())

Shape: (284807, 31)
Fraud cases: 492


# Prepare the data

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [14]:
X = fd.drop('Class', axis=1)
y = fd['Class']

In [15]:
# split the data 
X_train , X_test , y_train , y_test = train_test_split(
    X,y , test_size=0.2 , random_state=42, stratify=y
)

In [16]:
# scalarAmount and Time only 
scalar  = StandardScaler()
X_train[['Amount','Time']] = scalar.fit_transform(X_train[['Amount','Time']])
X_test[['Amount','Time']] = scalar.transform(X_test[['Amount','Time']])

In [17]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train frauds:", y_train.sum())
print("Test frauds:", y_test.sum())

Train shape: (227845, 30)
Test shape: (56962, 30)
Train frauds: 394
Test frauds: 98


# Model

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix ,precision_recall_curve

In [19]:
# Baseline - no imbalance handling 

In [20]:
baseline  = LogisticRegression(max_iter=1000, random_state=42)

In [21]:
baseline.fit(X_train,y_train)
y_pred_baseline = baseline.predict(X_test)

In [22]:
print("Baseline model (no imbalance handling)")
print("="*50)
print(classification_report(y_test,y_pred_baseline,target_names=['Normal','Fraud']))

Baseline model (no imbalance handling)
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.83      0.64      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



In [23]:
# Class weights - tell model fraud costs more 
weighted = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

In [25]:
weighted.fit(X_train , y_train)
y_pred_weighted = weighted.predict(X_test)

In [26]:
print("Class Weighted Model")
print("="*50)
print(classification_report(y_test,y_pred_weighted,target_names=['Normal', 'Fraud']))

Class Weighted Model
              precision    recall  f1-score   support

      Normal       1.00      0.98      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



# Threshold tuning 

In [38]:
# get probabilities instead of prediction 

y_proba = weighted.predict_proba(X_test)[:,1]

thresholds = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
y_test_array = np.array(y_test).ravel()

In [39]:
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Frauds Caught'}")
print("-" * 60)

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int).ravel()
    from sklearn.metrics import precision_score, recall_score, f1_score
    p = precision_score(y_test_array, y_pred_thresh, zero_division=0)
    r = recall_score(y_test_array, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test_array, y_pred_thresh, zero_division=0)
    caught = y_pred_thresh[y_test == 1].sum()
    print(f"{thresh:<12} {p:<12.3f} {r:<12.3f} {f1:<12.3f} {caught}/98")

Threshold    Precision    Recall       F1           Frauds Caught
------------------------------------------------------------
0.1          0.008        0.949        0.016        93/98
0.2          0.016        0.939        0.032        92/98
0.3          0.027        0.918        0.053        90/98
0.4          0.042        0.918        0.080        90/98
0.5          0.061        0.918        0.114        90/98
0.6          0.087        0.908        0.158        89/98
0.7          0.121        0.908        0.214        89/98
0.8          0.161        0.898        0.273        88/98


In [37]:
print("y_proba shape:", y_proba.shape)
print("y_pred_thresh shape:", (y_proba >= 0.5).astype(int).ravel().shape)
print("y_test_array shape:", y_test_array.shape)

y_proba shape: (1, 2)
y_pred_thresh shape: (2,)
y_test_array shape: (56962,)


Small startup, 5 analysts  → Threshold 0.8
                            → Precision 0.161, catches 88/98
                            → Team can handle the volume

Large bank, 500 analysts   → Threshold 0.1  
                            → Recall 0.949, catches 93/98
                            → Missing fraud costs more than false alarms

# Random Forest with class weights

In [40]:
from sklearn.ensemble import RandomForestClassifier

In [41]:
rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [42]:
print("RANDOM FOREST MODEL")
print("="*50)
print(classification_report(y_test, y_pred_rf,
      target_names=['Normal', 'Fraud']))

RANDOM FOREST MODEL
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.96      0.74      0.84        98

    accuracy                           1.00     56962
   macro avg       0.98      0.87      0.92     56962
weighted avg       1.00      1.00      1.00     56962

